In [1]:

from pathlib import Path
import sys

# Project root directory
PROJECT_ROOT = Path(r"D:\Drone-research")

# Define important project locations
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Select yesterday's flight log
SELECTED_LOG = RAW_DIR / "2026-07-02 18-37-42.bin"

# Verify our environment
print("Python version:", sys.version.split()[0])
print("Project exists:", PROJECT_ROOT.exists())
print("Raw data directory exists:", RAW_DIR.exists())
print("Selected flight log exists:", SELECTED_LOG.exists())

if SELECTED_LOG.exists():
    print(
        "Selected log size:",
        round(SELECTED_LOG.stat().st_size / (1024 ** 2), 2),
        "MiB"
    )


Python version: 3.11.17
Project exists: True
Raw data directory exists: True
Selected flight log exists: True
Selected log size: 15.14 MiB


In [2]:

import pandas as pd
from pymavlink import mavutil

def extract_telemetry(log_path, message_type, instance=None):
    """
    Extract telemetry from an ArduPilot BIN log.
    """

    log = mavutil.mavlink_connection(str(log_path))
    records = []

    while True:
        # Read every message sequentially
        msg = log.recv_match()

        if msg is None:
            break

        # Filter message type manually
        if msg.get_type() != message_type:
            continue

        data = msg.to_dict()

        # Optional sensor instance filtering
        if instance is not None:
            sensor_id = data.get("I", data.get("IMU"))

            if sensor_id != instance:
                continue

        records.append(data)

    df = pd.DataFrame(records)

    if not df.empty and "TimeUS" in df.columns:
        df = df.sort_values(
            "TimeUS"
        ).reset_index(drop=True)

    return df


In [3]:

att_df = extract_telemetry(
    SELECTED_LOG,
    "ATT"
)

print("Total attitude records:", len(att_df))
print("Available columns:", att_df.columns.tolist())

display(att_df.head())


Total attitude records: 11437
Available columns: ['mavpackettype', 'TimeUS', 'DesRoll', 'Roll', 'DesPitch', 'Pitch', 'DesYaw', 'Yaw', 'ErrRP', 'ErrYaw', 'AEKF']


,mavpackettype,TimeUS,DesRoll,Roll,DesPitch,Pitch,DesYaw,Yaw,ErrRP,ErrYaw,AEKF
0,ATT,23845225,0.0,1.24,0.0,-0.47,0.03,0.03,0.0,1.0,3
1,ATT,23945401,0.0,1.24,0.0,-0.47,0.02,0.02,0.0,1.0,3
2,ATT,24047397,0.0,1.24,0.0,-0.47,0.03,0.02,0.0,1.0,3
3,ATT,24147365,0.0,1.24,0.0,-0.47,0.02,0.02,0.0,1.0,3
4,ATT,24247300,0.0,1.25,0.0,-0.47,0.02,0.02,0.0,1.0,3


In [5]:

# Extract telemetry using our reusable function

gps_df = extract_telemetry(SELECTED_LOG, "GPS", instance=0)
imu_df = extract_telemetry(SELECTED_LOG, "IMU", instance=0)
baro_df = extract_telemetry(SELECTED_LOG, "BARO", instance=0)
arm_df = extract_telemetry(SELECTED_LOG, "ARM")

# Display dataset information
datasets = {
    "ATT": att_df,
    "GPS": gps_df,
    "IMU": imu_df,
    "BARO": baro_df,
    "ARM": arm_df
}

for name, df in datasets.items():
    print(
        f"{name}: {len(df)} records, "
        f"{len(df.columns)} columns"
    )


ATT: 11437 records, 11 columns
GPS: 5718 records, 16 columns
IMU: 28591 records, 16 columns
BARO: 11437 records, 11 columns
ARM: 18 records, 6 columns


In [6]:

# Check which sensor instances are present

for name in ["GPS", "IMU", "BARO"]:
    df = datasets[name]

    print(f"\n{name} sensor instances:")

    if "I" in df.columns:
        print(df["I"].value_counts().sort_index())
    else:
        print("No instance column found")



GPS sensor instances:
I
0    5718
Name: count, dtype: int64

IMU sensor instances:
I
0    28591
Name: count, dtype: int64

BARO sensor instances:
I
0    11437
Name: count, dtype: int64


In [7]:

import pandas as pd

quality_reports = []

for name, df in datasets.items():

    # Count missing values
    missing_values = df.isnull().sum().sum()

    # Count duplicate timestamps
    duplicate_times = (
        df["TimeUS"].duplicated().sum()
        if "TimeUS" in df.columns else 0
    )

    # Count invalid timestamps
    invalid_times = (
        (df["TimeUS"] <= 0).sum()
        if "TimeUS" in df.columns else 0
    )

    quality_reports.append({
        "Dataset": name,
        "Records": len(df),
        "Missing Values": missing_values,
        "Duplicate Timestamps": duplicate_times,
        "Invalid Timestamps": invalid_times
    })

quality_df = pd.DataFrame(quality_reports)

display(quality_df)


,Dataset,Records,Missing Values,Duplicate Timestamps,Invalid Timestamps
0,ATT,11437,0,0,0
1,GPS,5718,0,0,0
2,IMU,28591,0,0,0
3,BARO,11437,0,0,0
4,ARM,18,0,0,0


In [8]:

from pathlib import Path

# Create the processed data directory
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Use a unique name for our selected flight log
log_id = SELECTED_LOG.stem.replace(" ", "_")

# Export each telemetry dataset
for name, df in datasets.items():

    output_path = (
        PROCESSED_DIR /
        f"{log_id}_{name.lower()}.csv"
    )

    df.to_csv(output_path, index=False)

    print(
        f"Saved: {output_path.name} "
        f"({len(df)} records)"
    )

print("\nAll telemetry datasets exported successfully!")


Saved: 2026-07-02_18-37-42_att.csv (11437 records)
Saved: 2026-07-02_18-37-42_gps.csv (5718 records)
Saved: 2026-07-02_18-37-42_imu.csv (28591 records)
Saved: 2026-07-02_18-37-42_baro.csv (11437 records)
Saved: 2026-07-02_18-37-42_arm.csv (18 records)

All telemetry datasets exported successfully!
